# Adaptation Potential Metrics

This notebook creates separate downloadable CSVs for the five supported Adaptation Potential cards. The underlying FLOPROS, nature-based solution, and river-network calculations are unchanged; their results are reshaped into tidy, pivot-friendly card tables.

## 0. Setup

Country settings, administrative level, source paths, nominal NbS cell area, and the coastal assignment limit come from the selected country configuration. Shared calculations and card formatters live in `src/national_tool_metrics/sections/adaptation_potential.py`.

In [ ]:
from pathlib import Path
import importlib
import sys

WORKING_DIRECTORY = Path.cwd().resolve()
REPO_ROOT = WORKING_DIRECTORY.parent if WORKING_DIRECTORY.name == "notebooks" else WORKING_DIRECTORY
SRC_DIRECTORY = REPO_ROOT / "src"
if str(SRC_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SRC_DIRECTORY))

from national_tool_metrics import load_country_config
from national_tool_metrics.boundaries import load_admin_boundaries
from national_tool_metrics.outputs import write_card_output
import national_tool_metrics.sections.adaptation_potential as adaptation_section

importlib.reload(adaptation_section)
from national_tool_metrics.sections.adaptation_potential import (
    ADAPTATION_POTENTIAL_CARD_DIMENSIONS,
    ADAPTATION_POTENTIAL_CARD_OPTIONAL_DIMENSIONS,
    assemble_adaptation_potential_card_metrics,
    build_flopros_metrics,
    build_nbs_metrics,
    build_river_network_context_metrics,
)

In [ ]:
config = load_country_config("MOZ", repo_root=REPO_ROOT)
admin_regions = load_admin_boundaries(config)

print(f"Country: {config.country.name} ({config.country.iso3})")
print(f"Administrative level: {config.country.admin_level.upper()}")
print(f"Administrative regions: {len(admin_regions):,}")
print(f"Nominal NbS cell area: {config.parameters['nbs_nominal_cell_area_ha']} ha")
print(f"Maximum coastal assignment distance: {config.parameters['max_coastal_assignment_distance_m']:,} m")

## 1. Existing Flood Protection — FLOPROS

Calculate the modal positive FLOPROS return-period value in each administrative region. Zero-valued cells are treated as no-data, and regions without positive cells retain a blank no-data value; where more than one positive value has the same highest frequency, the lower return period is selected.

In [ ]:
flopros_metrics = build_flopros_metrics(config, admin_regions)
flopros_metrics.head()

## 2. Nature-based Solution Potential

Summarize slope vegetation, mangrove, and river catchment restoration opportunities. Area and per-hectare totals use the source documentation's nominal 9-arcsecond cell assumption of 6.25 hectares. Mangrove cells outside administrative polygons are assigned to their nearest region only within the configured 5 km cap. Biodiversity is averaged over valid opportunity cells; cost and carbon metrics are summed over valid opportunity cells. Slope-vegetation costs and benefits are also broken down into other land cover, crops, and bare ground; mangrove metrics are broken down into accreting, static or moderately retreating, and fast-retreating shoreline conditions. River catchment restoration remains a total because its opportunity raster is binary.

In [ ]:
nbs_metrics = build_nbs_metrics(config, admin_regions)
nbs_metrics.head()

## 3. River Network Context

Clip the river network to each administrative region and classify its length using the grid-cell urbanisation raster. Classes 10 to 13 are grouped as rural (including water), classes 21 to 23 as town, and class 30 as city. Small raster no-data gaps at boundaries use the nearest valid class within 5 km. The three classified lengths are required to sum to total river length.

In [ ]:
river_network_context_metrics = build_river_network_context_metrics(
    config,
    admin_regions,
)
river_network_context_metrics.head()

## 4. Assemble Card Tables

Reshape the existing calculations into five tidy card tables. Nature-based solution cards expose their adjustable category, metric, and implementation-approach parameters as columns; Flood Protection cards use the same standard identifiers and subsection structure.

In [ ]:
card_metrics = assemble_adaptation_potential_card_metrics(
    config,
    admin_regions,
    flopros_metrics,
    nbs_metrics,
    river_network_context_metrics,
)

for card, frame in card_metrics.items():
    print(f"{card}: {len(frame):,} rows")
next(iter(card_metrics.values())).head()

## 5. Export

Write one canonical downloadable CSV for each supported Adaptation Potential card after reviewing the tables above.

In [ ]:
output_paths = {}
for card, frame in card_metrics.items():
    output_paths[card] = write_card_output(
        frame,
        config,
        section="adaptation_potential",
        card=card,
        dimension_columns=ADAPTATION_POTENTIAL_CARD_DIMENSIONS[card],
        optional_dimension_columns=ADAPTATION_POTENTIAL_CARD_OPTIONAL_DIMENSIONS[card],
    )
    print(f"Exported {card} metrics to: {output_paths[card]}")